In [ ]:
# Cell 1 — Setup: load ATT&CK catalogue + LM Studio connectivity check

import json, re, requests

# ── 1. Load ATT&CK STIX ───────────────────────────────────────────
ATTACK_JSON = "OSRs/ATTACK/enterprise-attack-v16.1.json"
with open(ATTACK_JSON, encoding="utf-8") as f:
    stix_bundle = json.load(f)

objects = stix_bundle.get("objects", [])

# ── 2. Build technique catalogue ──────────────────────────────────
parent_map  = {}   # sub-tcode → parent tcode
revoked_map = {}   # old tcode → new tcode

def get_tcode(obj):
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id", "")
    return ""

# Index all objects by id
stix_by_id = {o["id"]: o for o in objects}

# Build maps
for o in objects:
    if o.get("type") != "relationship":
        continue
    src = stix_by_id.get(o.get("source_ref",""))
    tgt = stix_by_id.get(o.get("target_ref",""))
    if not src or not tgt:
        continue
    tc_src = get_tcode(src)
    tc_tgt = get_tcode(tgt)
    rt = o.get("relationship_type","")
    if rt == "subtechnique-of" and tc_src and tc_tgt:
        parent_map[tc_src] = tc_tgt
    elif rt == "revoked-by" and tc_src and tc_tgt:
        revoked_map[tc_src] = tc_tgt

def resolve_to_parent(tc):
    tc = revoked_map.get(tc, tc)
    tc = parent_map.get(tc, tc)
    tc = revoked_map.get(tc, tc)
    return tc

# Active parent techniques only
revoked_tcodes = set(revoked_map.keys())
for o in objects:
    if o.get("type") == "attack-pattern":
        if o.get("x_mitre_revoked") or o.get("revoked"):
            tc = get_tcode(o)
            if tc:
                revoked_tcodes.add(tc)

technique_ids   = []   # list of T-codes
technique_names = {}   # T-code → name
technique_descs = {}   # T-code → description

for o in objects:
    if o.get("type") != "attack-pattern":
        continue
    if o.get("x_mitre_revoked") or o.get("revoked"):
        continue
    tc = get_tcode(o)
    if not tc or "." in tc:   # skip sub-techniques
        continue
    if tc in revoked_tcodes:
        continue
    technique_ids.append(tc)
    technique_names[tc] = o.get("name", "")
    technique_descs[tc] = o.get("description", "")

technique_ids = sorted(set(technique_ids))
technique_id_set = set(technique_ids)

print(f"Active parent techniques: {len(technique_ids)}")
print(f"Sample: {technique_ids[:5]}")

# ── 3. Build technique list string for prompt ─────────────────────
technique_list_str = "\n".join(
    f"{tc}: {technique_names[tc]}"
    for tc in technique_ids
)
print(f"\nTechnique catalogue sample:")
print("\n".join(technique_list_str.split("\n")[:5]))
print("...")

# ── 4. LM Studio config ───────────────────────────────────────────
LM_STUDIO_URL = "http://localhost:1234/v1/chat/completions"
LM_MODEL      = "google/gemma-4-26b-a4b"   # label for logging only
TOP_K         = 5

SYSTEM_PROMPT = """You are a cybersecurity expert specializing in MITRE ATT&CK threat intelligence.

Your task: given a CVE vulnerability description, identify the most relevant ATT&CK techniques an adversary would use to exploit it.

Rules:
- You MUST only select techniques from the provided catalogue. Do not invent T-codes.
- Return EXACTLY a JSON object with one key "techniques" containing a list of T-codes, ranked by relevance (most relevant first).
- Return NO other text, explanation, or markdown. Only the raw JSON object.
- Select between 1 and 5 techniques. Only include techniques that are clearly relevant.

Output format (strictly follow this):
{"techniques": ["T1190", "T1059", "T1078"]}"""

def build_user_prompt(cve_description, db_hint=None):
    prompt = (f"CVE Description:\n{cve_description}\n\n"
              f"Valid ATT&CK Technique Catalogue (T-code: Name):\n"
              f"{technique_list_str}\n\n"
              f"Identify the top-{TOP_K} most relevant ATT&CK parent "
              f"techniques for this CVE.")
    if db_hint:
        prompt += (f"\n\nHint: A vulnerability database suggests these "
                   f"techniques may be relevant: {db_hint}\n"
                   f"Use this as a guide but apply your own judgment.")
    return prompt

def call_llm(cve_description, db_hint=None, retries=2):
    payload = {
        "model"      : LM_MODEL,
        "messages"   : [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(
                                    cve_description, db_hint)},
        ],
        "temperature": 0.0,
        "max_tokens" : 4096,
        "stream"     : False,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    for attempt in range(retries + 1):
        try:
            resp    = requests.post(LM_STUDIO_URL, json=payload, timeout=90)
            resp.raise_for_status()
            content = resp.json()["choices"][0]["message"]["content"].strip()
            content = re.sub(r'^```json\s*|^```\s*|```$', '',
                             content, flags=re.MULTILINE).strip()
            parsed  = json.loads(content)
            raw_tcs = parsed.get("techniques", [])
            valid   = [tc.strip().upper() for tc in raw_tcs
                       if tc.strip().upper() in technique_id_set]
            return valid, content
        except requests.exceptions.ConnectionError:
            print("✗ Cannot connect — is LM Studio server running on port 1234?")
            return [], ""
        except json.JSONDecodeError:
            if attempt < retries:
                import time; time.sleep(1)
                continue
            found = re.findall(r'T\d{4}', content)
            valid = [tc.upper() for tc in found
                     if tc.upper() in technique_id_set]
            return valid[:TOP_K], content
        except Exception as e:
            print(f"  Error: {e}")
            return [], ""

# ── 5. Connectivity check ─────────────────────────────────────────
print("\nTesting LM Studio connection...")
test_tcs, test_raw = call_llm(
    "SQL injection vulnerability allows unauthenticated remote "
    "attackers to execute arbitrary SQL commands via user input.")
if test_tcs:
    print(f"✓ Connected.")
    print(f"  Predicted : {test_tcs}")
    print(f"  Raw output: {test_raw}")
else:
    print(f"✗ Failed. Raw: {test_raw}")
    print("  → Start LM Studio → Local Server → Load model → Start server")

Active parent techniques: 214
Sample: ['T1001', 'T1003', 'T1005', 'T1006', 'T1007']

Technique catalogue sample:
T1001: Data Obfuscation
T1003: OS Credential Dumping
T1005: Data from Local System
T1006: Direct Volume Access
T1007: System Service Discovery
...

Testing LM Studio connection...
✓ Connected.
  Predicted : ['T1190', 'T1210', 'T1059', 'T1202', 'T1659']
  Raw output: {"techniques": ["T1190", "T1210", "T1059", "T1202", "T1659"]}


In [5]:
# Cell 1b — Debug raw LM Studio response

import requests, json

payload = {
    "model"      : LM_MODEL,
    "messages"   : [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_prompt(
            "SQL injection vulnerability allows unauthenticated remote "
            "attackers to execute arbitrary SQL commands via user input.")},
    ],
    "temperature": 0.0,
    "max_tokens" : 150,
    "stream"     : False,
    "chat_template_kwargs": {"enable_thinking": False},
}

try:
    resp = requests.post(LM_STUDIO_URL, json=payload, timeout=90)
    print(f"HTTP status  : {resp.status_code}")
    print(f"Raw response :\n{resp.text[:2000]}")
except Exception as e:
    print(f"Exception: {e}")

HTTP status  : 200
Raw response :
{
  "id": "chatcmpl-nqtn06v1jgc991gjfc2ilp",
  "object": "chat.completion",
  "created": 1775176396,
  "model": "qwen3.5-9b",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "",
        "reasoning_content": "Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Input: A CVE description (\"SQL injection vulnerability allows unauthenticated remote attackers to execute arbitrary SQL commands via user input.\").\n    *   Task: Identify the most relevant MITRE ATT&CK techniques from the provided catalogue.\n    *   Constraints:\n        *   Return EXACTLY a JSON object with one key \"techniques\".\n        *   List of T-codes, ranked by relevance (most relevant first).\n        *   No other text, explanation, or markdown. Only raw JSON.\n        *   Select between 1 and 5 techniques.\n        *   Only include clearly relevant techniques from the provided catalogue.\n\n2",
        "tool_calls": [

In [23]:
# Cell 1c — Fix: disable Qwen3 thinking mode + increase token budget

def build_user_prompt(cve_description, db_hint=None):
    # /nothink at start disables Qwen3 chain-of-thought
    prompt = (
              f"CVE Description:\n{cve_description}\n\n"
              f"Valid ATT&CK Technique Catalogue (T-code: Name):\n"
              f"{technique_list_str}\n\n"
              f"Identify the top-{TOP_K} most relevant ATT&CK parent "
              f"techniques for this CVE.")
    if db_hint:
        prompt += (f"\n\nHint: A vulnerability database suggests these "
                   f"techniques may be relevant: {db_hint}\n"
                   f"Use this as a guide but apply your own judgment.")
    return prompt

def call_llm(cve_description, db_hint=None, retries=2):
    payload = {
        "model"      : LM_MODEL,
        "messages"   : [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(
                                    cve_description, db_hint)},
        ],
        "temperature": 0.0,
        "max_tokens" : 5120,    # enough for JSON even with some thinking
        "stream"     : False,
    }
    for attempt in range(retries + 1):
        try:
            resp    = requests.post(LM_STUDIO_URL, json=payload, timeout=90)
            resp.raise_for_status()
            data    = resp.json()
            choice  = data["choices"][0]["message"]

            # Qwen3: actual answer may be in content or reasoning_content
            content = choice.get("content", "").strip()
            if not content:
                # fallback: sometimes JSON ends up in reasoning_content
                content = choice.get("reasoning_content", "").strip()
                # extract last JSON block from reasoning
                matches = re.findall(r'\{[^{}]+\}', content)
                content = matches[-1] if matches else ""

            content = re.sub(r'^```json\s*|^```\s*|```$', '',
                             content, flags=re.MULTILINE).strip()

            parsed  = json.loads(content)
            raw_tcs = parsed.get("techniques", [])
            valid   = [tc.strip().upper() for tc in raw_tcs
                       if tc.strip().upper() in technique_id_set]
            return valid, content

        except requests.exceptions.ConnectionError:
            print("✗ Cannot connect — is LM Studio server running?")
            return [], ""
        except json.JSONDecodeError:
            if attempt < retries:
                import time; time.sleep(1)
                continue
            found = re.findall(r'T\d{4}', content)
            valid = [tc.upper() for tc in found
                     if tc.upper() in technique_id_set]
            return valid[:TOP_K], content
        except Exception as e:
            print(f"  Error: {e}")
            return [], ""

# ── Retest ────────────────────────────────────────────────────────
print("Retesting with /nothink fix...")
test_tcs, test_raw = call_llm(
    """Apache Log4j2 2.0-beta9 through 2.15.0 (excluding security releases 2.12.2, 2.12.3, and 2.3.1)
    JNDI features used in configuration, log messages, and parameters do not protect against attacker 
    controlled LDAP and other JNDI related endpoints. An attacker who can control log messages or log
    message parameters can execute arbitrary code loaded from LDAP servers when message lookup substitution
    is enabled. From log4j 2.15.0, this behavior has been disabled by default. From version 2.16.0
    (along with 2.12.2, 2.12.3, and 2.3.1), this functionality has been completely removed.
    Note that this vulnerability is specific to log4j-core and does not affect log4net, log4cxx, or other Apache Logging Services projects. """
    )
print(f"Predicted : {test_tcs}")
print(f"Raw output: {test_raw}")

Retesting with /nothink fix...
Predicted : ['T1190', 'T1210', 'T1505', 'T1202', 'T1620']
Raw output: {"techniques": ["T1190", "T1210", "T1505", "T1202", "T1620"]}


In [25]:
# Cell 2 — Full KEV + SMET Evaluation with Gemma-4

import ast, time, glob
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path

LM_MODEL = "gemma-4-26b-a4b"   # update to exact name shown in LM Studio

# ══════════════════════════════════════════════════════════════════
# A. Load eval datasets
# ══════════════════════════════════════════════════════════════════

# ── KEV ───────────────────────────────────────────────────────────
with open("OSRs/kev-07.28.2025_attack-16.1-enterprise.json",
          encoding="utf-8") as f:
    kev_data = json.load(f)

kev_by_cve = {}
for entry in kev_data["mapping_objects"]:
    cve_id = entry.get("capability_id","").strip()
    tc_raw = entry.get("attack_object_id","").strip()
    desc   = entry.get("capability_description","").strip()
    if not cve_id.startswith("CVE-") or not tc_raw:
        continue
    parent_tc = parent_map.get(tc_raw, tc_raw)
    if "." in parent_tc or parent_tc not in technique_id_set:
        continue
    if cve_id not in kev_by_cve:
        kev_by_cve[cve_id] = {"desc": desc, "techs": set()}
    kev_by_cve[cve_id]["techs"].add(parent_tc)

# Enrich KEV with NVD descriptions
print("Loading NVD for KEV enrichment...")
nvd_lookup = {}
for fpath in sorted(glob.glob("OSRs/NVD/nvdcve-2.0-*.json")):
    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)
    for entry in data.get("vulnerabilities", []):
        cve_obj = entry.get("cve", {})
        cve_id  = cve_obj.get("id", "")
        for d in cve_obj.get("descriptions", []):
            if d.get("lang") == "en":
                txt = d.get("value","").strip()
                if txt and not txt.startswith("** REJECT"):
                    nvd_lookup[cve_id] = txt
                break
print(f"NVD loaded: {len(nvd_lookup):,}")

kev_cve_ids, kev_descs, kev_labels = [], [], []
for cve_id, val in kev_by_cve.items():
    if not val["techs"]:
        continue
    short = val["desc"]
    full  = nvd_lookup.get(cve_id, "")
    desc  = f"{short} {full}".strip() if full and len(full) > len(short) \
            else short
    if not desc:
        continue
    kev_cve_ids.append(cve_id)
    kev_descs.append(desc)
    kev_labels.append(val["techs"])

print(f"KEV eval rows: {len(kev_cve_ids)}")

# ── SMET ──────────────────────────────────────────────────────────
df_smet = pd.read_excel("CVE_annotated_dataset.xlsx", engine="openpyxl")

# name → T-code map
name_to_tc = {}
for tc, name in technique_names.items():
    name_to_tc[name.strip().lower()] = tc

# id2mitre fallback
try:
    import urllib.request
    with urllib.request.urlopen(
            "https://raw.githubusercontent.com/basel-a/SMET/main/id2mitre.json",
            timeout=10) as r:
        id2mitre = json.loads(r.read().decode("utf-8"))
    for k, v in id2mitre.items():
        if isinstance(v, str) and re.match(r'T\d{4}', v):
            parent_tc = parent_map.get(v, v)
            if parent_tc in technique_id_set:
                name_to_tc[k.strip().lower()] = parent_tc
    print(f"id2mitre loaded: {len(id2mitre)} entries")
except Exception as e:
    print(f"id2mitre fetch failed: {e}")

def parse_smet_techs(val):
    if pd.isna(val): return []
    try:    names = ast.literal_eval(str(val))
    except: names = [str(val)]
    return [name_to_tc[n.strip().lower()]
            for n in names if n.strip().lower() in name_to_tc]

df_smet["tcodes"] = df_smet["ATT&CK Techniques"].apply(parse_smet_techs)
df_smet_eval = df_smet[df_smet["tcodes"].apply(len) > 0].reset_index(drop=True)

smet_cve_ids = df_smet_eval["ID"].tolist()
smet_descs   = df_smet_eval["Description"].tolist()
smet_labels  = [set(tc) for tc in df_smet_eval["tcodes"].tolist()]

print(f"SMET eval rows: {len(df_smet_eval)}")

# ══════════════════════════════════════════════════════════════════
# B. LLM evaluation loop
# ══════════════════════════════════════════════════════════════════

def eval_llm(cve_ids, descs, labels, label, sample_n=None,
             use_db_hint=False, db=None, sleep_s=0.5):
    """
    Runs LLM on each CVE, computes R@1 / R@5 / R@10.
    sample_n: if set, randomly sample that many CVEs first.
    """
    if sample_n and sample_n < len(cve_ids):
        import random
        idx   = random.sample(range(len(cve_ids)), sample_n)
        cve_ids = [cve_ids[i] for i in idx]
        descs   = [descs[i]   for i in idx]
        labels  = [labels[i]  for i in idx]

    h1 = h5 = h10 = 0
    parse_failures = 0
    results = []

    for i, (cve_id, desc, true_labels) in enumerate(
            tqdm(zip(cve_ids, descs, labels),
                 total=len(cve_ids), desc=label)):

        db_hint = None
        if use_db_hint and db and cve_id in db:
            db_techs = db[cve_id].get("TECHNIQUES", [])
            if db_techs:
                names = [technique_names.get(tc, tc) for tc in db_techs]
                db_hint = ", ".join(names[:6])

        preds, raw = call_llm(desc, db_hint=db_hint)

        if not preds:
            parse_failures += 1

        hit1  = int(bool(preds) and preds[0] in true_labels)
        hit5  = int(bool(true_labels.intersection(set(preds[:5]))))
        hit10 = int(bool(true_labels.intersection(set(preds[:10]))))
        h1+=hit1; h5+=hit5; h10+=hit10

        results.append({
            "cve_id"      : cve_id,
            "predicted"   : preds,
            "true_labels" : list(true_labels),
            "hit1"        : hit1,
            "hit5"        : hit5,
            "hit10"       : hit10,
        })

        time.sleep(sleep_s)   # avoid hammering local server

    n = len(cve_ids)
    print(f"\n── {label} (n={n}) ──────────────────────────────────")
    print(f"  R@1  : {h1/n*100:.2f}%")
    print(f"  R@5  : {h5/n*100:.2f}%")
    print(f"  R@10 : {h10/n*100:.2f}%")
    print(f"  Parse failures: {parse_failures}/{n}")
    return {"r1":h1/n*100,"r5":h5/n*100,"r10":h10/n*100,
            "n":n, "failures":parse_failures, "rows":results}

# ── Load CVE2CAPEC DB for hint mode ───────────────────────────────
print("\nLoading CVE2CAPEC DB for hints...")
cve2capec_db = {}
for fpath in sorted(Path("OSRs/CVE2CAPEC").glob("*.jsonl")):
    with open(fpath, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                entry = json.loads(line)
            except: continue
            for cve_id, data in entry.items():
                raw_techs = data.get("TECHNIQUES", [])
                parent_tcs = set()
                for t in raw_techs:
                    t = t.strip()
                    if not t.upper().startswith("T"):
                        t = "T" + t
                    t = t.upper()
                    pt = parent_map.get(t, t)
                    if pt in technique_id_set:
                        parent_tcs.add(pt)
                cve2capec_db[cve_id] = {
                    "CWE"       : data.get("CWE", []),
                    "TECHNIQUES": list(parent_tcs),
                }
print(f"CVE2CAPEC loaded: {len(cve2capec_db):,} CVEs")

# ══════════════════════════════════════════════════════════════════
# C. Run evaluations
# (set sample_n=None to run full set — will take 20-40 min)
# ══════════════════════════════════════════════════════════════════
SAMPLE_N = None   # start with 50 per set to check speed/quality

print(f"\n{'='*60}")
print(f"LLM: {LM_MODEL}  |  sample_n={SAMPLE_N} per set")
print(f"{'='*60}")

# KEV — no hint
r_kev = eval_llm(kev_cve_ids, kev_descs, kev_labels,
                 f"KEV no hint", sample_n=SAMPLE_N)

# KEV — with DB hint
r_kev_hint = eval_llm(kev_cve_ids, kev_descs, kev_labels,
                      f"KEV + DB hint", sample_n=SAMPLE_N,
                      use_db_hint=True, db=cve2capec_db)

# SMET — no hint
r_smet = eval_llm(smet_cve_ids, smet_descs, smet_labels,
                  f"SMET no hint", sample_n=SAMPLE_N)

# SMET — with DB hint
r_smet_hint = eval_llm(smet_cve_ids, smet_descs, smet_labels,
                       f"SMET + DB hint", sample_n=SAMPLE_N,
                       use_db_hint=True, db=cve2capec_db)

# ══════════════════════════════════════════════════════════════════
# D. Comparison table
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("COMPARISON — LLM vs Best Previous Results")
print("="*70)
print(f"{'Method':<38} {'R@1':>8} {'R@5':>8} {'R@10':>8}")
print("─"*70)

rows = [
    # KEV
    ("KEV  | AttackBERT Rerank CE (best)",   13.13, 35.80, 54.65),
    ("KEV  | Gemma-4 no hint",
     r_kev['r1'],      r_kev['r5'],      r_kev['r10']),
    ("KEV  | Gemma-4 + DB hint",
     r_kev_hint['r1'], r_kev_hint['r5'], r_kev_hint['r10']),
    # SMET
    ("SMET | AttackBERT Hybrid RRF (best)",  32.45, 63.91, 76.16),
    ("SMET | Paper baseline (AttackBERT)",   32.45, 67.71, None),
    ("SMET | Gemma-4 no hint",
     r_smet['r1'],      r_smet['r5'],      r_smet['r10']),
    ("SMET | Gemma-4 + DB hint",
     r_smet_hint['r1'], r_smet_hint['r5'], r_smet_hint['r10']),
]

prev_set = None
for name, r1, r5, r10 in rows:
    cur = name[:4]
    if prev_set and prev_set != cur:
        print("─"*70)
    r10s = f"{r10:>7.2f}%" if r10 else "       —"
    print(f"{name:<38} {r1:>7.2f}% {r5:>7.2f}% {r10s}")
    prev_set = cur

print("="*70)
print(f"\nNote: LLM results on sample_n={SAMPLE_N} — "
      f"set SAMPLE_N=None for full run")

# ── Show a few examples ───────────────────────────────────────────
print("\n── Sample predictions (KEV, no hint) ───────────────────")
for row in r_kev["rows"][:3]:
    print(f"\n  CVE   : {row['cve_id']}")
    print(f"  Pred  : {row['predicted']}")
    print(f"  True  : {row['true_labels']}")
    print(f"  Hit@5 : {'✓' if row['hit5'] else '✗'}")

Loading NVD for KEV enrichment...
NVD loaded: 333,022
KEV eval rows: 419
id2mitre loaded: 594 entries
SMET eval rows: 302

Loading CVE2CAPEC DB for hints...
CVE2CAPEC loaded: 341,492 CVEs

LLM: gemma-4-26b-a4b  |  sample_n=None per set


KEV no hint: 100%|██████████| 419/419 [29:25<00:00,  4.21s/it]



── KEV no hint (n=419) ──────────────────────────────────
  R@1  : 51.79%
  R@5  : 78.28%
  R@10 : 78.28%
  Parse failures: 0/419


KEV + DB hint: 100%|██████████| 419/419 [29:37<00:00,  4.24s/it]



── KEV + DB hint (n=419) ──────────────────────────────────
  R@1  : 51.79%
  R@5  : 73.27%
  R@10 : 73.27%
  Parse failures: 0/419


SMET no hint: 100%|██████████| 302/302 [21:09<00:00,  4.20s/it]



── SMET no hint (n=302) ──────────────────────────────────
  R@1  : 37.75%
  R@5  : 74.17%
  R@10 : 74.17%
  Parse failures: 0/302


SMET + DB hint: 100%|██████████| 302/302 [21:32<00:00,  4.28s/it]


── SMET + DB hint (n=302) ──────────────────────────────────
  R@1  : 37.75%
  R@5  : 73.84%
  R@10 : 73.84%
  Parse failures: 0/302

COMPARISON — LLM vs Best Previous Results
Method                                      R@1      R@5     R@10
──────────────────────────────────────────────────────────────────────
KEV  | AttackBERT Rerank CE (best)       13.13%   35.80%   54.65%
KEV  | Gemma-4 no hint                   51.79%   78.28%   78.28%
KEV  | Gemma-4 + DB hint                 51.79%   73.27%   73.27%
──────────────────────────────────────────────────────────────────────
SMET | AttackBERT Hybrid RRF (best)      32.45%   63.91%   76.16%
SMET | Paper baseline (AttackBERT)       32.45%   67.71%        —
SMET | Gemma-4 no hint                   37.75%   74.17%   74.17%
SMET | Gemma-4 + DB hint                 37.75%   73.84%   73.84%

Note: LLM results on sample_n=None — set SAMPLE_N=None for full run

── Sample predictions (KEV, no hint) ───────────────────

  CVE   : CVE-2024-34102


# The Above R@* are not Recall (All answers are correct) they are Rate (At least one answer is correct)

In [26]:
# Cell 3 — Full metrics for Gemma-4 no hint

import numpy as np
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             label_ranking_average_precision_score,
                             label_ranking_loss, coverage_error)

def compute_full_metrics(rows, technique_ids, label, top_k=5):
    """
    rows: list of dicts with 'predicted' (ranked T-code list)
          and 'true_labels' (list/set of T-codes)
    Builds binary label matrix + inverse-rank score matrix,
    then computes all metrics.
    """
    tc_list  = sorted(technique_ids)
    tc_index = {tc: i for i, tc in enumerate(tc_list)}
    N        = len(rows)
    C        = len(tc_list)

    Y_true   = np.zeros((N, C), dtype=np.float32)
    Y_scores = np.zeros((N, C), dtype=np.float32)
    Y_pred   = np.zeros((N, C), dtype=np.int32)

    for i, row in enumerate(rows):
        # True labels
        for tc in row["true_labels"]:
            if tc in tc_index:
                Y_true[i, tc_index[tc]] = 1.0

        # Predicted — inverse rank scores
        for rank, tc in enumerate(row["predicted"]):
            if tc in tc_index:
                Y_scores[i, tc_index[tc]] = 1.0 / (rank + 1)
                if rank < top_k:
                    Y_pred[i, tc_index[tc]] = 1

    # ── Filter: only keep classes with ≥1 positive in this split ──
    active_cols = np.where(Y_true.sum(0) > 0)[0]
    Y_t = Y_true[:, active_cols]
    Y_s = Y_scores[:, active_cols]
    Y_p = Y_pred[:, active_cols]

    # ── Ranking metrics (use full score matrix) ────────────────────
    # Need at least one positive per row for LRAP
    valid_rows = Y_t.sum(1) > 0
    lrap = label_ranking_average_precision_score(
        Y_t[valid_rows], Y_s[valid_rows])
    rl   = label_ranking_loss(Y_t[valid_rows], Y_s[valid_rows])
    ce   = coverage_error(Y_t[valid_rows], Y_s[valid_rows])

    # ── Per-sample classification metrics ─────────────────────────
    # Weighted F1/P/R per sample, then mean ± SD
    sample_p, sample_r, sample_f1 = [], [], []
    for i in range(N):
        if Y_t[i].sum() == 0:
            continue
        p  = precision_score(Y_t[i], Y_p[i], zero_division=0)
        r  = recall_score(Y_t[i],    Y_p[i], zero_division=0)
        f  = f1_score(Y_t[i],        Y_p[i], zero_division=0)
        sample_p.append(p); sample_r.append(r); sample_f1.append(f)

    wp   = np.mean(sample_p);  wp_sd  = np.std(sample_p)
    wr   = np.mean(sample_r);  wr_sd  = np.std(sample_r)
    wf1  = np.mean(sample_f1); wf1_sd = np.std(sample_f1)

    # ── Macro F1 across active classes ────────────────────────────
    # per-class F1 then average (only over classes with ≥1 true pos)
    class_f1s = []
    for c in range(Y_t.shape[1]):
        if Y_t[:, c].sum() == 0:
            continue
        cf1 = f1_score(Y_t[:, c], Y_p[:, c], zero_division=0)
        class_f1s.append(cf1)
    macro_f1    = np.mean(class_f1s)
    macro_f1_sd = np.std(class_f1s)

    # ── Recall@K ──────────────────────────────────────────────────
    def r_at_k(k):
        hits, total = 0, 0
        for row in rows:
            pos = set(row["true_labels"]) & set(tc_list)
            if not pos: continue
            hits  += len(pos & set(row["predicted"][:k]))
            total += len(pos)
        return hits / total if total else 0.0

    print(f"\n{'='*60}")
    print(f"FULL METRICS — {label}")
    print(f"{'='*60}")
    print(f"  N (CVEs evaluated)     : {N}")
    print(f"  Active classes         : {len(active_cols)}")
    print(f"\n  ── Ranking Metrics ──────────────────────────────")
    print(f"  LRAP                   : {lrap:.4f}")
    print(f"  Ranking Loss           : {rl:.4f}")
    print(f"  Coverage Error         : {ce:.4f}")
    print(f"\n  ── Retrieval Metrics ────────────────────────────")
    for k in [1, 3, 5, 10]:
        print(f"  R@{k:<3}                  : {r_at_k(k)*100:.2f}%")
    print(f"\n  ── Classification Metrics (mean ± SD) ───────────")
    print(f"  Weighted Precision     : {wp:.4f}  ±{wp_sd:.4f}")
    print(f"  Weighted Recall        : {wr:.4f}  ±{wr_sd:.4f}")
    print(f"  Weighted F1            : {wf1:.4f}  ±{wf1_sd:.4f}")
    print(f"  Macro F1               : {macro_f1:.4f}  ±{macro_f1_sd:.4f}")

    return dict(lrap=lrap, ranking_loss=rl, coverage_error=ce,
                r1=r_at_k(1), r5=r_at_k(5), r10=r_at_k(10),
                weighted_p=wp, weighted_p_sd=wp_sd,
                weighted_r=wr, weighted_r_sd=wr_sd,
                weighted_f1=wf1, weighted_f1_sd=wf1_sd,
                macro_f1=macro_f1, macro_f1_sd=macro_f1_sd)

# ── Run for Gemma-4 no hint ───────────────────────────────────────
m_kev  = compute_full_metrics(
    r_kev["rows"],  technique_ids, f"KEV  — Gemma-4 no hint (n={r_kev['n']})")
m_smet = compute_full_metrics(
    r_smet["rows"], technique_ids, f"SMET — Gemma-4 no hint (n={r_smet['n']})")

# ── Side-by-side comparison table ────────────────────────────────
print(f"\n{'='*72}")
print("SIDE-BY-SIDE: KEV vs SMET — Gemma-4 no hint")
print(f"{'='*72}")
print(f"{'Metric':<28} {'KEV':>20} {'SMET':>20}")
print("─"*72)
metrics_display = [
    ("LRAP",             "lrap",          False),
    ("Ranking Loss",     "ranking_loss",  False),
    ("Coverage Error",   "coverage_error",False),
    ("R@1",             "r1",            True),
    ("R@3",             None,            True),
    ("R@5",             "r5",            True),
    ("R@10",            "r10",           True),
    ("Weighted P",      "weighted_p",    False),
    ("Weighted R",      "weighted_r",    False),
    ("Weighted F1",     "weighted_f1",   False),
    ("Macro F1",        "macro_f1",      False),
]
for label, key, is_pct in metrics_display:
    if key is None:
        # R@3 computed inline
        def r3(rows):
            hits=tot=0
            for row in rows:
                pos = set(row["true_labels"]) & set(technique_ids)
                if not pos: continue
                hits += len(pos & set(row["predicted"][:3]))
                tot  += len(pos)
            return hits/tot if tot else 0
        kv = r3(r_kev["rows"])*100
        sv = r3(r_smet["rows"])*100
        print(f"  {'R@3':<26} {kv:>19.2f}% {sv:>19.2f}%")
        continue
    kv = m_kev[key]
    sv = m_smet[key]
    ksd_key = key + "_sd"
    ssd_key = key + "_sd"
    if ksd_key in m_kev:
        kstr = f"{kv*100:.2f}% ±{m_kev[ksd_key]*100:.2f}%" if is_pct \
               else f"{kv:.4f} ±{m_kev[ksd_key]:.4f}"
        sstr = f"{sv*100:.2f}% ±{m_smet[ssd_key]*100:.2f}%" if is_pct \
               else f"{sv:.4f} ±{m_smet[ssd_key]:.4f}"
    else:
        kstr = f"{kv*100:.2f}%" if is_pct else f"{kv:.4f}"
        sstr = f"{sv*100:.2f}%" if is_pct else f"{sv:.4f}"
    print(f"  {label:<26} {kstr:>20} {sstr:>20}")
print("="*72)

# ── SMET paper comparison ─────────────────────────────────────────
print(f"\n── vs SMET paper (AttackBERT) ───────────────────────────")
print(f"  Coverage Error : {m_smet['coverage_error']:.4f}  "
      f"(paper: 13.96)")
print(f"  Ranking Loss   : {m_smet['ranking_loss']:.4f}  "
      f"(paper:  0.05)")
print(f"  LRAP           : {m_smet['lrap']:.4f}  "
      f"(paper: 53.77%)")
print(f"  R@5            : {m_smet['r5']*100:.2f}%  "
      f"(paper: 67.71%)")


FULL METRICS — KEV  — Gemma-4 no hint (n=419)
  N (CVEs evaluated)     : 419
  Active classes         : 108

  ── Ranking Metrics ──────────────────────────────
  LRAP                   : 0.3376
  Ranking Loss           : 0.5786
  Coverage Error         : 89.6444

  ── Retrieval Metrics ────────────────────────────
  R@1                    : 18.55%
  R@3                    : 29.32%
  R@5                    : 35.38%
  R@10                   : 35.38%

  ── Classification Metrics (mean ± SD) ───────────
  Weighted Precision     : 0.2299  ±0.1509
  Weighted Recall        : 0.4245  ±0.3290
  Weighted F1            : 0.2822  ±0.1858
  Macro F1               : 0.0814  ±0.1437

FULL METRICS — SMET — Gemma-4 no hint (n=302)
  N (CVEs evaluated)     : 302
  Active classes         : 40

  ── Ranking Metrics ──────────────────────────────
  LRAP                   : 0.5034
  Ranking Loss           : 0.3613
  Coverage Error         : 19.3013

  ── Retrieval Metrics ────────────────────────────
  R@

In [28]:
def compute_full_metrics(rows, technique_ids, label, top_k=5):
    tc_list  = sorted(technique_ids)
    tc_index = {tc: i for i, tc in enumerate(tc_list)}
    N        = len(rows)
    C        = len(tc_list)

    Y_true   = np.zeros((N, C), dtype=np.float32)
    Y_scores = np.zeros((N, C), dtype=np.float32)
    Y_pred   = np.zeros((N, C), dtype=np.int32)

    for i, row in enumerate(rows):
        for tc in row["true_labels"]:
            if tc in tc_index:
                Y_true[i, tc_index[tc]] = 1.0
        for rank, tc in enumerate(row["predicted"]):
            if tc in tc_index:
                Y_scores[i, tc_index[tc]] = 1.0 / (rank + 1)
                if rank < top_k:
                    Y_pred[i, tc_index[tc]] = 1

    # ── Ranking metrics: use FULL 214-class space ─────────────────
    # (do NOT filter to active_cols — that distorts the ranking)
    valid_rows = Y_true.sum(1) > 0
    lrap = label_ranking_average_precision_score(
        Y_true[valid_rows], Y_scores[valid_rows])
    rl   = label_ranking_loss(Y_true[valid_rows], Y_scores[valid_rows])
    ce   = coverage_error(Y_true[valid_rows], Y_scores[valid_rows])

    # ── Classification metrics: filter to active classes only ─────
    active_cols = np.where(Y_true.sum(0) > 0)[0]
    Y_t = Y_true[:, active_cols]
    Y_p = Y_pred[:, active_cols]

    sample_p, sample_r, sample_f1 = [], [], []
    for i in range(N):
        if Y_t[i].sum() == 0:
            continue
        p = precision_score(Y_t[i], Y_p[i], zero_division=0)
        r = recall_score(Y_t[i],    Y_p[i], zero_division=0)
        f = f1_score(Y_t[i],        Y_p[i], zero_division=0)
        sample_p.append(p); sample_r.append(r); sample_f1.append(f)

    wp   = np.mean(sample_p);  wp_sd  = np.std(sample_p)
    wr   = np.mean(sample_r);  wr_sd  = np.std(sample_r)
    wf1  = np.mean(sample_f1); wf1_sd = np.std(sample_f1)

    class_f1s = []
    for c in range(Y_t.shape[1]):
        if Y_t[:, c].sum() == 0:
            continue
        cf1 = f1_score(Y_t[:, c], Y_p[:, c], zero_division=0)
        class_f1s.append(cf1)
    macro_f1    = np.mean(class_f1s)
    macro_f1_sd = np.std(class_f1s)

    def r_at_k(k):
        hits, total = 0, 0
        for row in rows:
            pos = set(row["true_labels"]) & set(tc_list)
            if not pos: continue
            hits  += len(pos & set(row["predicted"][:k]))
            total += len(pos)
        return hits / total if total else 0.0

    print(f"\n{'='*60}")
    print(f"FULL METRICS — {label}")
    print(f"{'='*60}")
    print(f"  N                      : {N}")
    print(f"  Active classes         : {len(active_cols)}")
    print(f"  Full label space       : {C}")
    print(f"\n  ── Ranking Metrics (full {C}-class space) ────────")
    print(f"  LRAP                   : {lrap:.4f}")
    print(f"  Ranking Loss           : {rl:.4f}")
    print(f"  Coverage Error         : {ce:.4f}")
    print(f"\n  ── Retrieval Metrics ────────────────────────────")
    for k in [1, 3, 5]:
        print(f"  R@{k:<3}                  : {r_at_k(k)*100:.2f}%")
    print(f"\n  ── Classification (mean ± SD, {len(active_cols)} active classes)")
    print(f"  Weighted Precision     : {wp:.4f}  ±{wp_sd:.4f}")
    print(f"  Weighted Recall        : {wr:.4f}  ±{wr_sd:.4f}")
    print(f"  Weighted F1            : {wf1:.4f}  ±{wf1_sd:.4f}")
    print(f"  Macro F1               : {macro_f1:.4f}  ±{macro_f1_sd:.4f}")

    return dict(lrap=lrap, ranking_loss=rl, coverage_error=ce,
                r1=r_at_k(1), r3=r_at_k(3), r5=r_at_k(5),
                weighted_p=wp, weighted_p_sd=wp_sd,
                weighted_r=wr, weighted_r_sd=wr_sd,
                weighted_f1=wf1, weighted_f1_sd=wf1_sd,
                macro_f1=macro_f1, macro_f1_sd=macro_f1_sd)

m_kev  = compute_full_metrics(
    r_kev["rows"],  technique_ids,
    f"KEV  — Gemma-4 no hint (n={r_kev['n']})")
m_smet = compute_full_metrics(
    r_smet["rows"], technique_ids,
    f"SMET — Gemma-4 no hint (n={r_smet['n']})")


FULL METRICS — KEV  — Gemma-4 no hint (n=419)
  N                      : 419
  Active classes         : 108
  Full label space       : 214

  ── Ranking Metrics (full 214-class space) ────────
  LRAP                   : 0.3268
  Ranking Loss           : 0.5771
  Coverage Error         : 177.1838

  ── Retrieval Metrics ────────────────────────────
  R@1                    : 18.55%
  R@3                    : 29.32%
  R@5                    : 35.38%

  ── Classification (mean ± SD, 108 active classes)
  Weighted Precision     : 0.2299  ±0.1509
  Weighted Recall        : 0.4245  ±0.3290
  Weighted F1            : 0.2822  ±0.1858
  Macro F1               : 0.0814  ±0.1437

FULL METRICS — SMET — Gemma-4 no hint (n=302)
  N                      : 302
  Active classes         : 40
  Full label space       : 214

  ── Ranking Metrics (full 214-class space) ────────
  LRAP                   : 0.4471
  Ranking Loss           : 0.3539
  Coverage Error         : 99.0331

  ── Retrieval Metrics ──